# Data Ingestion Pipeline POC

This section contains various POCs of the ingestion pipeline where the application needs to ingest user-provided PDFs, DOCX, PPTs and maybe _(if time permits)_ OCR for PDFs. You can find here quickly prototyped working POC that were meant to be part of the final application.

## Installing Dependencies

For the type of ingestion we're planning to choose, `docling` over a faster & lightweight custom pipeline implementation of `PyMuPDF`, `python-docx`, and `python-pptx` due to time constraints and clear abstraction of handling multiple source formats.

In [1]:
# Using FastAPI to emulate HTTP request behaviour
!/root/.local/bin/uv add fastapi uvicorn python-multipart

# msgspec for hanlding json
!/root/.local/bin/uv add msgspec

# Installing docling for actual ingestion
!/root/.local/bin/uv add docling

Resolved 158 packages in 1ms
Checked 124 packages in 38ms
Resolved 158 packages in 1ms
Checked 124 packages in 39ms
Resolved 158 packages in 1ms
Checked 124 packages in 41ms


## Helper Code

This section contains the helper code that initialises the backend server to emulate the behaviour of HTTP requests. Just for this POC we're using Plain HTML that will be embedded into Jupyter Notebook, and FastAPI acts as the backend.

### Constants

In [2]:
HOST = "127.0.0.1"
PORT = 8000
BASE_URL = f"http://{HOST}:{PORT}"                 # Python requests only
FRONTEND_REQUEST_URL = "../proxy/8000/upload" # browser fetch
FRONTEND_HEALTH_URL = "../proxy/8000/health"       # browser fetch
FRONTEND_UI_TITLE = "Upload Legacy and Modern PDFs"

### Frontend Embedded UI

Use `start_frontend` function to view the embedded HTML frontend

In [3]:
HTML_TEMPLATE = r"""
<div style="font-family: sans-serif; padding: 16px; max-width: 800px;">
  <h2 id="uiTitle"></h2>

  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin-bottom: 16px;">
    <div style="border: 1px solid #ccc; padding: 12px; border-radius: 8px;">
      <label><strong>Legacy PDF</strong></label><br><br>
      <input type="file" id="legacyPdf" accept=".pdf" />
    </div>

    <div style="border: 1px solid #ccc; padding: 12px; border-radius: 8px;">
      <label><strong>Modern PDF</strong></label><br><br>
      <input type="file" id="modernPdf" accept=".pdf" />
    </div>
  </div>

  <button id="uploadBtn"
          style="padding: 10px 16px; border: none; border-radius: 8px; background: #0b5fff; color: white; cursor: pointer;">
    Upload and Ingest
  </button>

  <div id="status" style="margin-top: 16px; white-space: pre-wrap; font-size: 14px;"></div>

  <hr style="margin: 24px 0;" />

  <h3>Legacy Markdown</h3>
  <pre id="legacyOut" style="background: #f7f7f7; padding: 12px; border-radius: 8px; max-height: 300px; overflow: auto;"></pre>

  <h3>Modern Markdown</h3>
  <pre id="modernOut" style="background: #f7f7f7; padding: 12px; border-radius: 8px; max-height: 300px; overflow: auto;"></pre>
</div>

<script>
(() => {
  const API_URL = __API_URL__;
  const UI_TITLE = __UI_TITLE__;

  const titleEl = document.getElementById("uiTitle");
  const uploadBtn = document.getElementById("uploadBtn");
  const legacyInput = document.getElementById("legacyPdf");
  const modernInput = document.getElementById("modernPdf");
  const status = document.getElementById("status");
  const legacyOut = document.getElementById("legacyOut");
  const modernOut = document.getElementById("modernOut");

  titleEl.textContent = UI_TITLE;
  console.log("[FRONTEND] Widget loaded");
  console.log("[FRONTEND] API URL:", API_URL);

  uploadBtn.onclick = async () => {
    console.log("Upload Button Clicked");

    const legacyFile = legacyInput.files[0];
    const modernFile = modernInput.files[0];

    legacyOut.textContent = "";
    modernOut.textContent = "";

    if (!legacyFile || !modernFile) {
      status.textContent = "Please choose both PDFs.";
      return;
    }

    const formData = new FormData();
    formData.append("legacy_pdf", legacyFile);
    formData.append("modern_pdf", modernFile);

    status.textContent = "Sending PDFs to FastAPI...";
    console.log("[FRONTEND] Sending request");

    try {
      const res = await fetch(API_URL, {
        method: "POST",
        body: formData
      });

      status.textContent = "Response received. HTTP " + res.status;
      console.log("[FRONTEND] HTTP status:", res.status);

      const text = await res.text();
      console.log("[FRONTEND] Raw response:", text);

      if (!res.ok) {
        status.textContent = "FastAPI returned error:\\n" + text;
        return;
      }

      const data = JSON.parse(text);
      status.textContent = "Ingestion complete.";

      legacyOut.textContent = (data.legacy && data.legacy.markdown) || "";
      modernOut.textContent = (data.modern && data.modern.markdown) || "";
    } catch (err) {
      console.error("[FRONTEND] Request failed:", err);
      status.textContent = "Request failed:\\n" + String(err);
    }
  };
})();
</script>
"""

In [4]:
import os
from IPython.display import HTML
import msgspec

def start_frontend():
    # JupyterHub sets JUPYTERHUB_SERVICE_PREFIX to the correct base path,
    # e.g. "/jupyter-hack-team-2743-260611030959-efd753de"
    prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "jupyter-hack-team-2743-260611030959-efd753de").rstrip("/")
    request_url = f"{prefix}/proxy/{PORT}/upload" if prefix else FRONTEND_REQUEST_URL

    html = (
        HTML_TEMPLATE
        .replace("__API_URL__", msgspec.json.encode(request_url).decode("utf-8"))
        .replace("__UI_TITLE__", msgspec.json.encode(FRONTEND_UI_TITLE).decode("utf-8"))
    )
    return HTML(html)

## Backend

In [5]:
from __future__ import annotations

import threading
import time
from pathlib import Path
from typing import Any

import uvicorn
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response  # <-- add Response

In [6]:
async def save_upload_file_to_temp(upload: UploadFile, out_dir: Path) -> Path:
    if not upload.filename:
        raise HTTPException(status_code=400, detail="Missing filename")

    if not upload.filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400, detail=f"Only PDF files are allowed: {upload.filename}",
        )

    out_path = out_dir / upload.filename

    with out_path.open("wb") as f:
        while True:
            chunk = await upload.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)

    await upload.close()
    return out_path

In [7]:
def create_helper_fastapi_app() -> FastAPI:
    app = FastAPI(title="Docling Helper Backend")

    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],
        allow_credentials=True,
        allow_methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"],
        allow_headers=["*"],
    )

    @app.get("/health")
    async def health():
        return {"ok": True}

    return app


class NotebookUvicornServer(uvicorn.Server):
    def install_signal_handlers(self):
        return

In [8]:
class BackendServer:
    def __init__(self):
        self.server = None
        self.thread = None
        self.app = create_helper_fastapi_app()

    def start(self, host: str = HOST, port: int = PORT):
        self.app = self.app if self.app else create_helper_fastapi_app()
        config = uvicorn.Config(app=self.app, host=host, port=port, log_level="info")
        self.server = NotebookUvicornServer(config=config)

        self.thread = threading.Thread(target=self.server.run, daemon=True)
        self.thread.start()

        while not self.server.started:
            time.sleep(0.05)

        print(f"Helper backend running at http://{host}:{port}")
        print(f"Health URL: http://{host}:{port}/health")

    def stop(self, join_timeout: int = 5):
        if self.server is not None:
            self.server.should_exit = True
        if self.thread is not None:
            self.thread.join(timeout=join_timeout)
        print("Helper backend stopped.")

In [9]:
import requests

mock_instance = BackendServer()
mock_instance.start()

resp = requests.get(f"{BASE_URL}/health")
print(resp.status_code, resp.json())

mock_instance.stop()

INFO:     Started server process [13031]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Helper backend running at http://127.0.0.1:8000
Health URL: http://127.0.0.1:8000/health
INFO:     127.0.0.1:54256 - "GET /health HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [13031]


200 {'ok': True}
Helper backend stopped.


## Plain Text-based PDFs

We expect this to be the primary source of input file content, with table and MathJax based on the document content. Everything needs to be ingested.

In [10]:
from concurrent.futures import ProcessPoolExecutor

import msgspec
import tempfile
from docling.document_converter import DocumentConverter

In [11]:
class IngestedPdf(msgspec.Struct, frozen=True):
    file_name: str
    source_path: str
    markdown: str
    char_count: int

In [12]:
def ingest_single_pdf_with_docling(pdf_path: str | Path) -> IngestedPdf:
    

    path = Path(pdf_path).expanduser().resolve()

    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {path}")

    if path.suffix.lower() != ".pdf":
        raise ValueError(f"Only PDF files are supported. Got: {path.name}")

    converter = DocumentConverter()
    doc = converter.convert(str(path)).document
    markdown = doc.export_to_markdown()

    return IngestedPdf(
        file_name=path.name,
        source_path=str(path),
        markdown=markdown,
        char_count=len(markdown),
    )

In [13]:
def ingest_two_pdfs_with_docling_parallel(
    legacy_pdf_path: str | Path,
    modern_pdf_path: str | Path,
    max_workers: int = 2,
) -> dict[str, IngestedPdf]:
    legacy_pdf_path = str(Path(legacy_pdf_path).expanduser().resolve())
    modern_pdf_path = str(Path(modern_pdf_path).expanduser().resolve())

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            "legacy": executor.submit(ingest_single_pdf_with_docling, legacy_pdf_path),
            "modern": executor.submit(ingest_single_pdf_with_docling, modern_pdf_path),
        }

        return {name: future.result() for name, future in futures.items()}

In [14]:
backed_app = BackendServer()

In [15]:
@backed_app.app.post("/upload")
async def upload_documents(
    legacy_pdf: UploadFile = File(...),
    modern_pdf: UploadFile = File(...),
):
    with tempfile.TemporaryDirectory(prefix="docling_nb_") as tmp:
        tmp_dir = Path(tmp)

        legacy_path = await save_upload_file_to_temp(legacy_pdf, tmp_dir)
        modern_path = await save_upload_file_to_temp(modern_pdf, tmp_dir)

        result = ingest_two_pdfs_with_docling_parallel(
            legacy_pdf_path=legacy_path,
            modern_pdf_path=modern_path,
            max_workers=2,
        )

    # JSONResponse uses json.dumps which can't serialize msgspec.Struct.
    # Use msgspec.json.encode directly instead.
    return Response(
        content=msgspec.json.encode(result),
        media_type="application/json",
    )

In [16]:
backed_app.start()

INFO:     Started server process [13031]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Helper backend running at http://127.0.0.1:8000
Health URL: http://127.0.0.1:8000/health


In [17]:
start_frontend()

In [18]:
backed_app.stop()

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [13031]


Helper backend stopped.
